In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet('./data/taxi_data.parquet')
df = df[df['total_amount'] > 0] #Remove bad data (negative fares and zero fares)
df = df[df['trip_distance'] > 0] #Remove bad data (negative distances and zero distances)
df = df[df['payment_type'] != 2] #Remove cash payments
df = df[df['fare_amount'] > 0] #Remove bad data (negative fares and zero fares)
df = df[df['mta_tax'] >= 0] # Remove bad data (negative MTA tax)
df = df[df['tip_amount'] >= 0] # Remove bad data (negative tips)
df = df[df['tolls_amount'] >= 0] # Remove bad data (negative tolls)
df = df[df['extra'] >= 0] # Remove bad data (negative extra charges)
df.head()

,VendorID,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,is_yellow,ehail_fee,trip_type
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.8,1.0,N,140,236,1.0,...,0.5,3.75,0.0,1.0,18.75,2.5,0.0,True,NaN,NaN
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.7,1.0,N,236,79,1.0,...,0.5,3.00,0.0,1.0,31.30,2.5,0.0,True,NaN,NaN
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.4,1.0,N,79,211,1.0,...,0.5,2.00,0.0,1.0,17.00,2.5,0.0,True,NaN,NaN
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.8,1.0,N,211,148,1.0,...,0.5,3.20,0.0,1.0,16.10,2.5,0.0,True,NaN,NaN
5,1,2024-01-01 00:54:08,2024-01-01 01:26:31,1.0,4.7,1.0,N,148,141,1.0,...,0.5,6.90,0.0,1.0,41.50,2.5,0.0,True,NaN,NaN


In [3]:
def classify_tip(pct):
    if pct < 0.15:
        return 'Low'
    elif 0.15 <= pct <= 0.22:
        return 'Middle'
    else:
        return 'High'

df['tip_pct'] = df['tip_amount'] / (df['total_amount'] - df['tip_amount']) #Calculate tip percentage
df['tip_class'] = df['tip_pct'].apply(classify_tip)
df.drop(columns=['tip_pct', 'tip_amount', 'total_amount'], inplace=True)

In [4]:
df.columns

Index(['VendorID', 'pickup_datetime', 'dropoff_datetime', 'passenger_count',
       'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID',
       'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax',
       'tolls_amount', 'improvement_surcharge', 'congestion_surcharge',
       'Airport_fee', 'is_yellow', 'ehail_fee', 'trip_type', 'tip_class'],
      dtype='object')

In [5]:
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df['dropoff_datetime'] = pd.to_datetime(df['dropoff_datetime'])

# remove wrong datetime entries only 2024 data is kept
df = df[(df['pickup_datetime'].dt.year == 2024)]
df = df[(df['dropoff_datetime'].dt.year == 2024)]
df = df[df['dropoff_datetime'] > df['pickup_datetime']]

df['pickup_hour'] = df['pickup_datetime'].dt.hour
df['day_of_week'] = df['pickup_datetime'].dt.dayofweek

# Calculate duration in minutes
df['duration_min'] = (df['dropoff_datetime'] - df['pickup_datetime']).dt.total_seconds() / 60

df = df[df['duration_min'] > 0]  # Remove trips with non-positive duration

In [6]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 40.7834,
	"longitude": -73.9663,
	"start_date": "2024-01-01",
	"end_date": "2024-12-31",
	"hourly": ["temperature_2m", "apparent_temperature", "rain", "snowfall", "precipitation", "wind_speed_10m", "weather_code"],
	"timezone": "America/New_York",
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_apparent_temperature = hourly.Variables(1).ValuesAsNumpy()
hourly_rain = hourly.Variables(2).ValuesAsNumpy()
hourly_snowfall = hourly.Variables(3).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(4).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(5).ValuesAsNumpy()
hourly_weather_code = hourly.Variables(6).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["apparent_temperature"] = hourly_apparent_temperature
hourly_data["rain"] = hourly_rain
hourly_data["snowfall"] = hourly_snowfall
hourly_data["precipitation"] = hourly_precipitation
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["weather_code"] = hourly_weather_code

weather_df = pd.DataFrame(data = hourly_data)
weather_df['date'] = weather_df['date'].dt.tz_convert('America/New_York').dt.tz_localize(None)

Coordinates: 40.808433532714844°N -74.0198974609375°E
Elevation: 45.0 m asl
Timezone: b'America/New_York'b'GMT-5'
Timezone difference to GMT+0: -18000s


In [7]:
# add weather data 3/10 / 02:00:00
miss = [{
    'date': pd.to_datetime('2024-03-10 02:00:00'),
    'temperature_2m': 7.1,
    'apparent_temperature': 5,
    'rain': 0.0,
    'snowfall': 0.0,
    'precipitation': 0.0,
    'wind_speed_10m': 8.2,
    'weather_code': 45
},
    {
    'date': pd.to_datetime('2025-01-01 00:00:00'),
    'temperature_2m': 6.8,
    'apparent_temperature': 4.3,
    'rain': 0.0,
    'snowfall': 0.0,
    'precipitation': 0.0,
    'wind_speed_10m': 9.2,
    'weather_code': 3
}
]

weather_df = pd.concat([weather_df, pd.DataFrame(miss)], ignore_index=True)

In [8]:
df['weather_key'] = df['pickup_datetime'].dt.round('h')

df = pd.merge(
    df,
    weather_df,
    left_on='weather_key',
    right_on='date',
    how='left'
)

df = df.drop(columns=['weather_key', 'date'])

In [9]:
df[df['temperature_2m'].isna()]

,VendorID,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,pickup_hour,day_of_week,duration_min,temperature_2m,apparent_temperature,rain,snowfall,precipitation,wind_speed_10m,weather_code


In [10]:
df['Airport_fee'] = df['Airport_fee'].fillna(0)
df['congestion_surcharge'] = df['congestion_surcharge'].fillna(0)

In [11]:
df['speed_mph'] = (df['trip_distance'] / (df['duration_min'] + 0.001)) * 60

#remove unrealistic speeds
df = df[df['speed_mph'] <= 100]

In [12]:
cols_to_drop = [
    'store_and_fwd_flag',
    'vendor_id',
    'payment_type',
    'pickup_datetime',
    'dropoff_datetime',
    'temperature_2m',
    'rain',
    'VendorID',
    'ehail_fee',
    'trip_type'
]

In [13]:
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [14]:
df.columns

Index(['passenger_count', 'trip_distance', 'RatecodeID', 'PULocationID',
       'DOLocationID', 'fare_amount', 'extra', 'mta_tax', 'tolls_amount',
       'improvement_surcharge', 'congestion_surcharge', 'Airport_fee',
       'is_yellow', 'tip_class', 'pickup_hour', 'day_of_week', 'duration_min',
       'apparent_temperature', 'snowfall', 'precipitation', 'wind_speed_10m',
       'weather_code', 'speed_mph'],
      dtype='object')

In [15]:
#is there any missing data left?
print(df.isna().sum())

passenger_count          3716040
trip_distance                  0
RatecodeID               3716040
PULocationID                   0
DOLocationID                   0
fare_amount                    0
extra                          0
mta_tax                        0
tolls_amount                   0
improvement_surcharge          0
congestion_surcharge           0
Airport_fee                    0
is_yellow                      0
tip_class                      0
pickup_hour                    0
day_of_week                    0
duration_min                   0
apparent_temperature           0
snowfall                       0
precipitation                  0
wind_speed_10m                 0
weather_code                   0
speed_mph                      0
dtype: int64


In [16]:
df.to_parquet('./data/taxi_data_preprocessed_missing.parquet', index=False)
df.sample(frac=0.05, random_state=42).to_parquet('./data/taxi_data_preprocessed_missing_sampled.parquet', index=False)